In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/leads.csv")
print(df.shape)
df.head()

df = df.replace("Select", np.nan)
null_pct = df.isnull().mean()
drop_cols = null_pct[null_pct > 0.4].index.tolist()
drop_cols += ["Prospect ID", "Lead Number"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown")

print(df.isnull().sum().sum(), "nulls remaining")

df.to_csv("../data/cleaned/leads_cleaned.csv", index=False)

# Encode categorical columns
df_encoded = pd.get_dummies(df, drop_first=True)

# Separate target from features
y = df_encoded["Converted"]
X = df_encoded.drop(columns=["Converted"])

# Scale numeric features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))

(9240, 37)
0 nulls remaining


C:\Users\ASUS\AppData\Local\Temp\ipykernel_9708\4086874922.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


(6468, 166) (2772, 166)
Converted
0    0.614564
1    0.385436
Name: proportion, dtype: float64


In [5]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip install xgboost


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

print("Logistic Regression:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("F1:", f1_score(y_test, y_pred_lr))

Logistic Regression:
Accuracy: 0.9321789321789322
F1: 0.9102196752626552


In [8]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost:")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1:", f1_score(y_test, y_pred_xgb))

XGBoost:
Accuracy: 0.9411976911976911
Precision: 0.9406037000973709
Recall: 0.9044943820224719
F1: 0.9221957040572792


In [9]:
import joblib
joblib.dump(xgb_model, "../model/xgb_model.pkl")
joblib.dump(scaler, "../model/scaler.pkl")
joblib.dump(list(X.columns), "../model/feature_columns.pkl")

['../model/feature_columns.pkl']

In [10]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["segment_id"] = kmeans.fit_predict(X_scaled)

segment_conversion = df.groupby("segment_id")["Converted"].mean().sort_values(ascending=False)
print(segment_conversion)

segment_id
1    0.926184
0    0.398180
2    0.233895
Name: Converted, dtype: float64


In [11]:
segment_map = {1: "Hot", 0: "Warm", 2: "Cold"}
df["segment_label"] = df["segment_id"].map(segment_map)

print(df["segment_label"].value_counts())

joblib.dump(kmeans, "../model/kmeans_model.pkl")
joblib.dump(segment_map, "../model/segment_map.pkl")

segment_label
Warm    5495
Cold    3027
Hot      718
Name: count, dtype: int64


['../model/segment_map.pkl']